**Interactive Jupyter notebook implementing the interpolation, Iwasawa power series approximation, Newton slope extraction, and computation of the discriminant of the distinguished polynomial for the degree 2 case ($\lambda=2$). Includes commented test runs.**  

For $\lambda=2$, the characteristic polynomial of degree 2 has either two equal slopes, or two distinct slopes $s_1 \ne s_2$ with $v_p(\operatorname{Disc}) = 2 \min(s_1, s_2)$.

In [1]:
def power_series_twist1_lambda2_disc(D, p):
    r"""
    Computes the Iwasawa power series associated to a quadratic Dirichlet character, 
    the Iwasawa power series approximation, Newton slope extraction, and ramification classification.

    Special case: twist i=1 and only power series with lambda=2 are output. 
    For lambda=2, the discriminant of the characteristic polynomial is computed.
        
    INPUT:
    - ``D`` -- Fundamental discriminant.
    - ``p`` -- Prime number p (odd). 
   
    OUTPUT: 
    - ``fp`` -- Approximating polynomial of the Iwasawa power series in Qp[x].
    - ``lambda_inv`` -- Lambda-invariant of the power series.
    - ``v0`` -- Valuation of the constant coefficient of the power series.
    - ``disc`` -- Discriminant of the characteristic polynomial of degree 2.

    AUTHOR: Heiko Knospe, 2026
    """
    
    K = Qp(p, prec=50)
    v = K.valuation()
    R = PolynomialRing(K, 'x')

    i = 1 # twist

    if not Integer(D).is_fundamental_discriminant():
        print("Not a fundamental discriminant")
        return None, None, None, None

    if (D>0 and i%2==1) or (D<0 and i%2==0):
        print("Trivial p-adic L-function")
        return None, None, None, None

    cond = abs(D)
        
    if gcd(cond, p) > 1:
        print("Not supported. gcd(|D|, p) > 1.")
        return None, None, None, None
   
    chip = kronecker(D, p)
    
    if chip == 1:
        print(f"p={p}, twist={i}, D={D} ==> Trivial zero case")
        return None, None, None, None

    l0 = pari.lfun(D,0)
    v0 = v(l0)
    
    if v0 < 1 :  # lambda=0
        return None, 0, v0, None

    lp = pari.lfun(D,1-p)
    # check lambda >= 2
    mycheck=K((1-chip*p^(p-1))*lp - (1-chip)*l0)
    vp=v(mycheck)
    if vp <= 1 :  # lambda=1
        return None, 1, v0, None
        
      
    # Compute special values of the p-adic L-function
    # Variable: T = (1 + cond * p)^(-j + 1) - 1
    points = [
        (K((1 + cond * p)**(-j + 1) - 1), pari.lfun(D, 1 - j) * (1 - chip * p**(j - 1))) 
        for j in range(p - 1 + i, 15 * p, p - 1)
    ]

    # Interpolate to obtain an approximating power series
    fp = R.lagrange_polynomial(points)
    v0 = v(fp[0])  # Valuation of constant coefficient
        
    # Compute lambda-invariant
    # Search for lambda up to 15
    # Large lambda values are very rare
    lambda_bound = 15
    j = 0
    while j <= lambda_bound:
        if v(fp[j]) <= 0: 
            break
        j += 1
    
    if j > 2:  # lambda>2
        return fp, j, v0, None 
        
    # now lambda=2
    
    slopes = []
    discstring = ''
    ramtype = [', unramified field extension', ', ramified field extension', ', trivial field extension']

    
    fR = fp.truncate(j + 1) 
    try:       
        slopes = fR.newton_slopes()[0:j]  # Slopes of distinguished polynomial
    except Exception:
        slopes = 'NA'
        
    # Further analyze slopes 
    # (Multiplicity 1 corresponds to a linear factor of the distinguished polynomial, hence a root in Zp)
    if isinstance(slopes, list):
        if len(slopes) == 2:
            if slopes[0] == slopes[1]:
                # Equal slopes: extract factor and compute discriminant
                myslope = slopes[0]            
            
                try:
                    g = fp.factor_of_slope(myslope)
                    Disc = g.discriminant()
                    vD = v(Disc)
                    if vD % 2 == 1:
                        rt = ramtype[1]  # Ramified quadratic extension
                    else:
                        # Unit part projected to GF(p)
                        u0 = Disc.unit_part().residue()  
                        if u0.is_square():
                            rt = ramtype[2]  # Splits in Qp (two linear roots)
                        else:
                            rt = ramtype[0]  # Unramified quadratic extension
                        
                    discstring += ', slope=' + str(myslope) + rt
                except Exception:
                        discstring += ', slope=' + str(myslope) + ' ***Error***'
            else:
                # Distinct slopes s1 != s2: roots are in Qp, v(Disc) = 2 * min(s1, s2)
                vD = 2 * min(slopes[0], slopes[1])                   
    
    print(f'p={p}, twist={i}, Fundamental Discriminant={D} ==> lambda={j}, v(const)={v0}, slopes={slopes}{discstring}, v(Disc)={vD}')   
    return fp, j, v0, vD

In [2]:
p=5 # prime
i=1 # Teichmuller twist
disc=[0]*30
vals=[0]*20
lamb=[0]*17

for D in range(-2,-10000,-1):    
    if Integer(D).is_fundamental_discriminant()==False or gcd(D,p)!=1:        
        continue
    if i==1 and kronecker(D,p)==1:
        continue
    fp, j, v0, d = power_series_twist1_lambda2_disc(D,p) 
    if d != None:
        disc[d]+=1
    if v0 != None:       
        vals[v0]+=1
    if j != None:
        lamb[j]+=1

print(f'\np={p}, twist={i}, \ndistribution-lambda=      {lamb}, \ndistribution-v(constant)= {vals}, \ndistribution-discrimiminant (for lambda=2)= {disc}' )     

p=5, twist=1, Fundamental Discriminant=-127 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-523 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-803 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-1688 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-1723 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-1928 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-2152 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=5, twist=1, Fundamental Discriminant=-2483

In [3]:
p=3 # prime
i=1 # Teichmuller twist
disc=[0]*30
vals=[0]*20
lamb=[0]*17

# allocate several gigabytes for very high conductors
# pari.allocatemem(16* 1024 * 1024 * 1024)

for D in range(-10**10,-10**10-1000,-1):    
    if Integer(D).is_fundamental_discriminant()==False or gcd(D,p)!=1:        
        continue
    if i==1 and kronecker(D,p)==1:
        continue
    fp, j, v0, d = power_series_twist1_lambda2_disc(D,p) 
    if d != None:
        disc[d]+=1
    if v0 != None:       
        vals[v0]+=1
    if j != None:
        lamb[j]+=1

print(f'\np={p}, twist={i}, \ndistribution-lambda=      {lamb}, \ndistribution-v(constant)= {vals}, \ndistribution-discrimiminant (for lambda=2)= {disc}' ) 

p=3, twist=1, Fundamental Discriminant=-10000000120 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=3, twist=1, Fundamental Discriminant=-10000000180 ==> lambda=2, v(const)=2, slopes=[1, 1], slope=1, unramified field extension, v(Disc)=4
p=3, twist=1, Fundamental Discriminant=-10000000183 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=3, twist=1, Fundamental Discriminant=-10000000219 ==> lambda=2, v(const)=2, slopes=[1, 1], slope=1, trivial field extension, v(Disc)=2
p=3, twist=1, Fundamental Discriminant=-10000000339 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=3, twist=1, Fundamental Discriminant=-10000000363 ==> lambda=2, v(const)=2, slopes=[1, 1], slope=1, ramified field extension, v(Disc)=3
p=3, twist=1, Fundamental Discriminant=-10000000408 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2, ramified field extension, v(Disc)=1
p=3, twis